# Task 2: Data Cleaning & Validation

## Step 1: Identify and Remove Duplicate Rows

In this step, we check for duplicate records in the dataset and remove them to ensure data accuracy and consistency.

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("../data/Campaign_Raw.csv")

# Initial shape
print("Initial shape: ", df.shape)

# Preview data
df.head()

Initial shape:  (10028, 26)


,Data Source name,Date,Campaign Name,Campaign Effective Status,Ad Set Name,Ad Name,Country Funnel,Geo Location Segment,FB Spent Funnel (INR),Amount Spent (INR),...,Checkouts Initiated,Adds of Payment Info,Purchases,Purchases Conversion Value (INR),Website Contacts,Messaging Conversations Started,Adds to Cart Conversion Value (INR),Checkouts Initiated Conversion Value (INR),Adds of Payment Info Conversion Value (INR),Row Count
0,Brand A,02-01-2026,NaN,PAUSED,TOF | LAL 3-5% | Sales | Womens,Custom | Video 2 | EOSS | 15th Dec 25 – Copy,India,India,0.0,0.0,...,-0.0,0.0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,1.0
1,Brand A,03-01-2026,Growify | 11th June | TOF | ABO | INDIA | LAL ...,paused,TOF | LAL 3-5% | Sales | Womens,Custom | Video 2 | EOSS | 15th Dec 25 – Copy,India,India,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,NaN,1.0
2,NaN,20-01-2026,Growify | 11th June | TOF | ABO | INDIA | LAL ...,PAUSED,NaN,NaN,INDIA,India,0.0,0.0,...,0.0,0.0,0.0,-0.0,NaN,0.0,0.0,0.0,0.0,1.0
3,Brand A,01-01-2026,Growify | 11th June | TOF | ABO | INDIA | LAL ...,NaN,TOF | LAL 3-5% | Sales | Womens,Custom | Video 2 | EOSS | 15th Dec 25 – Copy,India,India,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,0.0,5964500.0,0.0,0.0,1.0
4,Brand A,11-01-2026,GROWIFY | 11TH JUNE | TOF | ABO | INDIA | LA...,PAUSED,NaN,Custom | Video 2 | EOSS | 15th Dec 25 – Copy,NAN,India,0.0,0.0,...,0.0,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,1.0


In [2]:
df.columns

Index(['Data Source name', 'Date', 'Campaign Name',
       'Campaign Effective Status', 'Ad Set Name', 'Ad Name', 'Country Funnel',
       'Geo Location Segment', 'FB Spent Funnel (INR)', 'Amount Spent (INR)',
       'Clicks (all)', 'Impressions', 'Page Likes', 'Landing Page Views',
       'Link Clicks', 'Adds to Cart', 'Checkouts Initiated',
       'Adds of Payment Info', 'Purchases', 'Purchases Conversion Value (INR)',
       'Website Contacts', 'Messaging Conversations Started',
       'Adds to Cart Conversion Value (INR)',
       'Checkouts Initiated Conversion Value (INR)',
       'Adds of Payment Info Conversion Value (INR)', 'Row Count'],
      dtype='object')

In [3]:
# Check duplicates
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

Duplicate rows: 310


In [4]:
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

After removing duplicates: (9718, 26)


## Step 2: Standardizing Date Format

In this step, we convert the Date column into a proper datetime format and handle any invalid or inconsistent values.

In [5]:
# Reload raw dataset
df = pd.read_csv("../data/Campaign_Raw.csv")

# Check raw date values
df['Date'].head(10)

0    02-01-2026
1    03-01-2026
2    20-01-2026
3    01-01-2026
4    11-01-2026
5    21-01-2026
6    22-01-2026
7    18-01-2026
8    19-01-2026
9    17-01-2026
Name: Date, dtype: object

### Converting Date Column

The Date column is converted into datetime format using pandas.

Since the dataset may contain mixed formats, we use `dayfirst=True` and `errors='coerce'` to handle inconsistencies.

In [6]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# Check converted values
df['Date'].head(10)

0   2026-01-02
1   2026-01-03
2   2026-01-20
3   2026-01-01
4   2026-01-11
5   2026-01-21
6   2026-01-22
7   2026-01-18
8   2026-01-19
9   2026-01-17
Name: Date, dtype: datetime64[ns]

### Checking Invalid Dates

We count null values created during conversion, which indicate missing or invalid date entries.

In [7]:
invalid_dates = df['Date'].isnull().sum()
print("Invalid date rows:", invalid_dates)

Invalid date rows: 1073


### Inspecting Invalid Entries

We inspect rows where Date values are missing after conversion.

In [8]:
df[df['Date'].isnull()][['Date']].head(20)

,Date
11,NaT
30,NaT
42,NaT
61,NaT
94,NaT
98,NaT
114,NaT
125,NaT
126,NaT
135,NaT


### Observations

A large number of invalid date values were found.

These were primarily due to missing values (NaN or empty entries), not incorrect formats.

Valid dates were successfully converted, and missing values will be handled in the next step.

### Date Validation

The dataset contains only a single Date column, so start/end date validation is not applicable.

## Step 3: Handling Missing Values

In this step, we analyze missing values across the dataset and apply appropriate strategies based on the nature and importance of each column.

We avoid blindly dropping rows and instead use context-aware handling.

In [9]:
df.isnull().sum().sort_values(ascending=False)

Website Contacts                               2573
Checkouts Initiated Conversion Value (INR)     1626
Adds of Payment Info Conversion Value (INR)    1092
Adds of Payment Info                           1078
Date                                           1073
Checkouts Initiated                            1045
Page Likes                                      906
Country Funnel                                  796
FB Spent Funnel (INR)                           750
Row Count                                       678
Purchases                                       673
Link Clicks                                     672
Amount Spent (INR)                              670
Purchases Conversion Value (INR)                670
Landing Page Views                              669
Geo Location Segment                            668
Impressions                                     668
Clicks (all)                                    666
Messaging Conversations Started                 664
Adds to Cart

### Missing Value Strategy

Different strategies were applied based on column type:

- Date column: Missing values retained as NaT to avoid incorrect assumptions.
- Numeric columns: Missing values filled with 0, assuming no activity occurred.
- Categorical/text columns: Missing values replaced with 'unknown' for consistency.

This approach ensures minimal data loss while maintaining data integrity.

In [10]:
# Fill numeric columns with 0
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
df[numeric_cols] = df[numeric_cols].fillna(0)

# Fill object (text) columns with 'unknown'
object_cols = df.select_dtypes(include=['object']).columns
df[object_cols] = df[object_cols].fillna('unknown')

In [11]:
df.isnull().sum().sort_values(ascending=False)

Date                                           1073
Data Source name                                  0
Campaign Name                                     0
Campaign Effective Status                         0
Ad Set Name                                       0
Ad Name                                           0
Country Funnel                                    0
Geo Location Segment                              0
FB Spent Funnel (INR)                             0
Amount Spent (INR)                                0
Clicks (all)                                      0
Impressions                                       0
Page Likes                                        0
Landing Page Views                                0
Link Clicks                                       0
Adds to Cart                                      0
Checkouts Initiated                               0
Adds of Payment Info                              0
Purchases                                         0
Purchases Co

In [12]:
print("Remaining missing values:", df.isnull().sum().sum())

Remaining missing values: 1073


## Step 4: Recalculating Metrics (CTR, CPC, CPM, ROI)

In this step, we recalculate key performance metrics using source columns to ensure accuracy.

We also flag rows where original values differ from recalculated values.

In [13]:
df.columns

Index(['Data Source name', 'Date', 'Campaign Name',
       'Campaign Effective Status', 'Ad Set Name', 'Ad Name', 'Country Funnel',
       'Geo Location Segment', 'FB Spent Funnel (INR)', 'Amount Spent (INR)',
       'Clicks (all)', 'Impressions', 'Page Likes', 'Landing Page Views',
       'Link Clicks', 'Adds to Cart', 'Checkouts Initiated',
       'Adds of Payment Info', 'Purchases', 'Purchases Conversion Value (INR)',
       'Website Contacts', 'Messaging Conversations Started',
       'Adds to Cart Conversion Value (INR)',
       'Checkouts Initiated Conversion Value (INR)',
       'Adds of Payment Info Conversion Value (INR)', 'Row Count'],
      dtype='object')

### Metric Definitions

- CTR = Clicks / Impressions  
- CPC = Spend / Clicks  
- CPM = (Spend / Impressions) * 1000  
- ROI = Revenue / Spend  

We calculate these using available columns.

In [14]:
# Rename columns for simplicity (optional but helpful)
df = df.rename(columns={
    'Amount Spent (INR)': 'spend',
    'FB Spent (INR)': 'fb_spend',
    'Purchases Conversion Value (INR)': 'revenue',
    'Purchases': 'purchases'
})

In [15]:
# calculate CTR(Click Through Rate)
df['CTR'] = df['Link Clicks'] / df['Impressions']

# calculate CPC(Cost Per Click)
df['CPC'] = df['spend'] / df['Link Clicks']

# calculate CPM(Cost Per 1000 Impressions)
df['CPM'] = (df['spend'] / df['Impressions']) * 1000

# calculate ROI(Return on Investment)
df['ROI'] = df['revenue'] / df['spend']

# replace inf values
df.replace([float('inf'), -float('inf')], 0, inplace=True)

In [16]:
df[['CTR', 'CPC', 'CPM', 'ROI']].describe()

,CTR,CPC,CPM,ROI
count,8967.000000,8952.000000,9.045000e+03,8.517000e+03
mean,0.334793,386.375726,9.760163e+03,8.006640e+02
std,2.331329,2843.501465,5.208424e+04,6.865147e+04
min,-17.825243,-35341.550000,-4.445478e+05,-2.367592e+04
25%,0.000000,0.000000,5.211317e+00,0.000000e+00
50%,0.013187,5.325387,5.111944e+02,0.000000e+00
75%,0.046055,37.018692,1.273100e+03,0.000000e+00
max,108.000000,82065.480000,1.322005e+06,6.334615e+06


### Observations

Metrics were successfully calculated using source columns.

The dataset did not contain original CTR, CPC, CPM, or ROI columns. Therefore, comparison and flagging were not applicable.

The recalculated metrics provide a reliable basis for performance analysis.

## Step 5: Normalizing String Columns

In this step, we standardize string-based columns to ensure consistency in formatting.

This includes converting text to lowercase, trimming whitespace, and handling inconsistent naming.

In [17]:
df.select_dtypes(include='object').columns

Index(['Data Source name', 'Campaign Name', 'Campaign Effective Status',
       'Ad Set Name', 'Ad Name', 'Country Funnel', 'Geo Location Segment'],
      dtype='object')

### Cleaning Strategy

- Convert all text to lowercase  
- Remove leading/trailing spaces  
- Replace multiple spaces with single space  
- Standardize text formatting

In [18]:
# normalize all string columns
string_cols = df.select_dtypes(include='object').columns

for col in string_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.lower()
    )


In [19]:
df['Campaign Name'].value_counts().head()

Campaign Name
growify | tof | conversion | india                  1649
growify | tof | conversion | uae                    1086
growify | retarget | conversion | aus+can+uk+usa     852
growify | tof | messaging | uae                      691
growify | tof | conversion | us                      666
Name: count, dtype: int64

### Observations

String columns were standardized to ensure consistency across the dataset.

This helps prevent issues caused by inconsistent text formats such as case differences or extra spaces.

## Step 6: Loading Cleaned Data into SQL Database

In this step, we load the cleaned campaign dataset into a SQLite database.

SQLite is used for simplicity and portability. The cleaned data is stored as a table for further analysis.

The same workflow can be extended to PostgreSQL for production-level use.

In [28]:
import sqlite3

In [29]:
# create/connect to database
conn = sqlite3.connect("../sql/cleaned_campaigns.db")

In [30]:
# load dataframe into SQL table
df.to_sql("campaign_cleaned", conn, if_exists="replace", index=False)

10028

In [33]:
pd.read_sql("SELECT * FROM campaign_cleaned LIMIT 5", conn).columns

Index(['Data Source name', 'Date', 'Campaign Name',
       'Campaign Effective Status', 'Ad Set Name', 'Ad Name', 'Country Funnel',
       'Geo Location Segment', 'FB Spent Funnel (INR)', 'spend',
       'Clicks (all)', 'Impressions', 'Page Likes', 'Landing Page Views',
       'Link Clicks', 'Adds to Cart', 'Checkouts Initiated',
       'Adds of Payment Info', 'purchases', 'revenue', 'Website Contacts',
       'Messaging Conversations Started',
       'Adds to Cart Conversion Value (INR)',
       'Checkouts Initiated Conversion Value (INR)',
       'Adds of Payment Info Conversion Value (INR)', 'Row Count', 'CTR',
       'CPC', 'CPM', 'ROI'],
      dtype='object')

In [32]:
# verify table content
query = "SELECT * FROM campaign_cleaned LIMIT 5"
pd.read_sql(query, conn)

,Data Source name,Date,Campaign Name,Campaign Effective Status,Ad Set Name,Ad Name,Country Funnel,Geo Location Segment,FB Spent Funnel (INR),spend,...,Website Contacts,Messaging Conversations Started,Adds to Cart Conversion Value (INR),Checkouts Initiated Conversion Value (INR),Adds of Payment Info Conversion Value (INR),Row Count,CTR,CPC,CPM,ROI
0,brand a,2026-01-02 00:00:00,unknown,paused,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
1,brand a,2026-01-03 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
2,unknown,2026-01-20 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,unknown,unknown,india,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None
3,brand a,2026-01-01 00:00:00,growify | 11th june | tof | abo | india | lal ...,unknown,tof | lal 3-5% | sales | womens,custom | video 2 | eoss | 15th dec 25 – copy,india,india,0.0,0.0,...,0.0,0.0,5964500.0,0.0,0.0,1.0,None,None,None,None
4,brand a,2026-01-11 00:00:00,growify | 11th june | tof | abo | india | lal ...,paused,unknown,custom | video 2 | eoss | 15th dec 25 – copy,nan,india,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,None,None,None,None


In [24]:
# count rowa in table
pd.read_sql("SELECT COUNT(*) FROM campaign_cleaned", conn)

,COUNT(*)
0,10028


In [25]:
conn.close()